## Install `holidays`

Databricks Serverless does not ship the `holidays` package. This cell installs it and
restarts Python so `src.features` can import it. Takes about five seconds.


In [0]:
%pip install holidays -q
dbutils.library.restartPython()


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


# 03 — Silver: cleaned + temporally enriched

- Drops target-leakage and post-facto columns (actual times, taxi, delay causes).
- Drops 2020 (COVID anomaly).
- Preserves `arrival_delay` nulls (cancelled / diverted); Gold filters them.
- Adds calendar + US-federal-holiday features via a broadcast **date dimension**
  built from the unit-tested helpers in `src.features`.
- Enforces the row contract with Delta `CHECK` constraints, then compacts with
  `OPTIMIZE ... ZORDER`.

Idempotent: full overwrite.


In [0]:
import sys
import time
from datetime import date

sys.path.append("..")

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, dayofmonth, dayofweek, floor, quarter as spark_quarter,
    to_date, udf, weekofyear, when, year, month as spark_month,
)
from pyspark.sql.types import (
    DateType, IntegerType, StringType, StructField, StructType,
)

from src import config
from src.features import (
    build_date_dimension, check_holiday, check_holiday_period, check_near_holiday,
    get_season,
)


In [0]:
bronze = spark.table(config.BRONZE)
print(f"Bronze rows: {bronze.count():,}")


Bronze rows: 3,000,000


## Column reduction + rename

In [0]:
selected = (
    bronze
    .select(
        col("AIRLINE").alias("airline_name"),
        col("AIRLINE_CODE").alias("airline_code"),
        col("FL_NUMBER").cast("int").alias("fl_number"),
        col("ORIGIN").alias("origin_airport_code"),
        col("DEST").alias("destination_airport_code"),
        to_date(col("FL_DATE")).alias("flight_date"),
        col("CRS_DEP_TIME").cast("int").alias("crs_dep_time"),
        col("CRS_ARR_TIME").cast("int").alias("crs_arr_time"),
        col("CRS_ELAPSED_TIME").cast("double").alias("crs_elapsed_time"),
        col("DISTANCE").cast("double").alias("distance"),
        col("DEP_DELAY").cast("double").alias("dep_delay"),
        col("ARR_DELAY").cast("double").alias("arrival_delay"),
    )
)


## Filters

In [0]:
enriched = (
    selected
    .withColumn("flight_year", year(col("flight_date")))
    .filter(col("flight_year") != 2020)          # COVID anomaly
    .filter(col("flight_date").isNotNull())
)

print(f"After 2020 filter: {enriched.count():,}")


After 2020 filter: 2,520,650


## Temporal + holiday features — a broadcast join, not four UDFs

The obvious implementation is a Python UDF per flag, and that is what this notebook
did originally. It is also the classic Spark anti-pattern.

**Why it was slow.** A Python UDF cannot run in the JVM. Every row is serialised out
to a Python worker, evaluated, and serialised back — four times per row, across
roughly two million rows. The holiday lookups themselves are cheap; the crossing of
the JVM/Python boundary is not, and it happens eight million times.

**The observation that fixes it.** All four flags are functions of the *date* alone,
and there are only about two thousand distinct dates in the dataset. The same answers
were being recomputed roughly a thousand times each.

**The fix.** Materialise one row per calendar date in the driver — where the Python
cost is paid ~2,000 times instead of ~2,000,000 — and broadcast it. The join runs
entirely in the JVM, and because the dimension is a few hundred kilobytes it ships to
every executor with no shuffle.

`src.features.build_date_dimension` is unit-tested against the very helpers it
replaces (`test_agrees_with_the_udf_helpers_it_replaces` compares every day of 2019),
so this is a provable refactor rather than a rewrite that hopefully matches.

The `dayofweek` convention is the trap: Spark numbers 1=Sunday, Python's
`date.weekday()` numbers 0=Monday. Getting that wrong shifts the feature by a day and
is invisible in aggregates, so `spark_day_of_week` encodes it once and is pinned by
its own test.


In [0]:
# One row per calendar date spanning the data, built from the unit-tested helpers.
bounds = enriched.select(
    F.min("flight_date").alias("lo"), F.max("flight_date").alias("hi")
).first()
print(f"Date range in Silver: {bounds['lo']} -> {bounds['hi']}")

DATE_DIM_SCHEMA = StructType([
    StructField("flight_date", DateType(), False),
    StructField("flight_month", IntegerType(), False),
    StructField("day_of_week", IntegerType(), False),
    StructField("week_of_year", IntegerType(), False),
    StructField("day_of_month", IntegerType(), False),
    StructField("quarter", IntegerType(), False),
    StructField("is_weekend", IntegerType(), False),
    StructField("is_holiday", IntegerType(), False),
    StructField("is_near_holiday", IntegerType(), False),
    StructField("is_holiday_period", IntegerType(), False),
    StructField("season", StringType(), False),
])

dim_rows = build_date_dimension(bounds["lo"], bounds["hi"])
date_dim = spark.createDataFrame(dim_rows, schema=DATE_DIM_SCHEMA)
print(f"Date dimension: {len(dim_rows):,} rows "
      f"(vs {enriched.count():,} flight rows the UDFs would have touched)")

silver = (
    enriched
    .join(F.broadcast(date_dim), on="flight_date", how="left")
    .withColumn("dep_hour", floor(col("crs_dep_time") / 100))
    .withColumn("arr_hour", floor(col("crs_arr_time") / 100))
)

# A left join silently produces nulls if the dimension fails to cover a date.
# It is built from the observed min/max, so a gap here is a real defect.
uncovered = silver.filter(col("season").isNull()).count()
assert uncovered == 0, f"{uncovered:,} rows joined to no date-dimension row"
print("Date dimension covers every flight row.")


Date range in Silver: 2019-01-01 -> 2023-08-31
Date dimension: 1,704 rows (vs 2,520,650 flight rows the UDFs would have touched)
Date dimension covers every flight row.


### Does it actually help? Measure, don't assert.

"Broadcast joins are faster than UDFs" is a claim, and a README that makes it without
a number is asking to be doubted. Both implementations run below on the same sample
and the ratio is printed.

The sample keeps this honest *and* cheap: the point is the ratio, and paying for a
full second pass over two million rows to produce a number the sample already gives
would be exactly the kind of unbudgeted compute this project is supposed to avoid.


In [0]:
BENCH_ROWS = 200_000
bench = enriched.limit(BENCH_ROWS)
bench.count()   # materialise before timing so the cache load is not measured

# --- the original: four Python UDFs, one call per row per flag ----------------
season_udf = udf(get_season, StringType())
holiday_udf = udf(check_holiday, IntegerType())
near_holiday_udf = udf(check_near_holiday, IntegerType())
holiday_period_udf = udf(check_holiday_period, IntegerType())

t0 = time.perf_counter()
(
    bench
    .withColumn("flight_month", spark_month(col("flight_date")))
    .withColumn("day_of_week", dayofweek(col("flight_date")))
    .withColumn("week_of_year", weekofyear(col("flight_date")))
    .withColumn("day_of_month", dayofmonth(col("flight_date")))
    .withColumn("quarter", spark_quarter(col("flight_date")))
    .withColumn("is_weekend", when(col("day_of_week").isin(1, 7), 1).otherwise(0))
    .withColumn("is_holiday", holiday_udf(col("flight_date")))
    .withColumn("is_near_holiday", near_holiday_udf(col("flight_date")))
    .withColumn("is_holiday_period", holiday_period_udf(col("flight_date")))
    .withColumn("season", season_udf(col("flight_month")))
    .write.format("noop").mode("overwrite").save()
)
udf_seconds = time.perf_counter() - t0

# --- the replacement: one broadcast join --------------------------------------
t0 = time.perf_counter()
(
    bench
    .join(F.broadcast(date_dim), on="flight_date", how="left")
    .write.format("noop").mode("overwrite").save()
)
join_seconds = time.perf_counter() - t0

print(f"Sample: {BENCH_ROWS:,} rows")
print(f"  4 Python UDFs   : {udf_seconds:7.2f}s")
print(f"  broadcast join  : {join_seconds:7.2f}s")
print(f"  speedup         : {udf_seconds / max(join_seconds, 1e-9):7.2f}x")
print()
print("`.write.format('noop')` forces full execution without writing anything, so")
print("the timing measures the transformation rather than Spark's laziness or the")
print("cost of a sink.")


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/udf.py:103: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


Sample: 200,000 rows
  4 Python UDFs   :   37.79s
  broadcast join  :    1.38s
  speedup         :   27.29x

`.write.format('noop')` forces full execution without writing anything, so
the timing measures the transformation rather than Spark's laziness or the
cost of a sink.


## Congestion features — the 57.7% the decomposition pointed at

`02_eda` §5 decomposed delay minutes by cause: late aircraft 38.4%, NAS (airspace and ATC)
19.3%, carrier 36.2%, extreme weather 5.8%. And §5's cascade chart showed late-aircraft
share climbing from 15.2% of delay minutes before 09:00 to 44.4% after 18:00.

Both of those point the same way. Late aircraft is not an independent cause — it is delay
*propagating* through the day, and propagation is a congestion phenomenon. Together with NAS
that is **57.7% of all delay minutes**, and none of it needs data this project does not have.

The pre-departure model currently sees only static attributes of a scheduled flight: airline,
route, month, day of week, hour. Nothing tells it how busy the airport is at that moment or
how deep into the operating day the flight sits. That is why it plateaus around 0.62 while
every model family ties — there are no interactions left to find in static fields.

Four features, all derived from `crs_dep_time` and route metadata:

| Feature | What it measures |
|---|---|
| `sched_deps_origin_hour` | Departures scheduled from this origin in this clock hour |
| `dep_bank_density` | Departures scheduled within ±60 minutes at this origin |
| `dep_sequence_in_day` | This flight's position in its carrier's departure sequence from this origin that day |
| `schedule_padding` | `crs_elapsed_time` minus the route's median — how much slack the airline built in |

**Why these are leak-free.** Every one is computed from *scheduled* times, which are published
months ahead and are fixed before any of the day's operations happen. Nothing here reads an
actual time, an actual delay, or the label. That is the property that makes congestion the
right next feature family rather than merely a plausible one — the strongest available signal
that is also legitimately knowable at prediction time.

**`schedule_padding` is the airline's own opinion.** Carriers pad block times on routes they
know run late, so the gap between a route's scheduled duration and its median is the operator
telling you where it expects trouble. It costs one window function.

**`dep_sequence_in_day` is the closest available proxy for aircraft rotation.** This dataset
has no `TAIL_NUM`, so airframes cannot be chained directly. A carrier's fifth departure of the
day from a given airport is nonetheless far more likely to be flying an aircraft that already
inherited delay than its first.

One caveat stated rather than buried: the route median in `schedule_padding` is computed over
the whole table, so it is informed by rows that later land in the test year. It is a property
of the *schedule*, not of any outcome, so this does not leak the target — but a stricter
design would compute it on the CV window alone, and a reader is entitled to want that named.


In [0]:
from pyspark.sql import Window

# Scheduled departure time in minutes since midnight. HHMM is not linear in time,
# so every window below orders and ranges on this rather than on crs_dep_time.
silver = silver.withColumn(
    "dep_minutes",
    (F.floor(col("crs_dep_time") / 100) * 60 + (col("crs_dep_time") % 100)).cast("int"),
)

# 1. How many departures share this origin and clock hour.
origin_hour = Window.partitionBy("origin_airport_code", "flight_date", "dep_hour")

# 2. Departures within +/-60 minutes at the same origin. A range window over
#    dep_minutes, so it follows the clock rather than a fixed row count.
bank = (
    Window.partitionBy("origin_airport_code", "flight_date")
    .orderBy("dep_minutes")
    .rangeBetween(-60, 60)
)

# 3. Where this flight sits in its own carrier's departure sequence from this
#    airport that day. No TAIL_NUM in this dataset, so rotation cannot be chained
#    directly; sequence position is the available proxy.
carrier_day = (
    Window.partitionBy("origin_airport_code", "airline_code", "flight_date")
    .orderBy("dep_minutes")
)

# 4. Route median scheduled duration, for schedule padding.
route = Window.partitionBy("origin_airport_code", "destination_airport_code")

silver = (
    silver
    .withColumn("sched_deps_origin_hour", F.count("*").over(origin_hour))
    .withColumn("dep_bank_density", F.count("*").over(bank))
    .withColumn("dep_sequence_in_day", F.row_number().over(carrier_day))
    .withColumn("route_median_elapsed",
                F.percentile_approx("crs_elapsed_time", 0.5).over(route))
    .withColumn("schedule_padding",
                col("crs_elapsed_time") - col("route_median_elapsed"))
    .drop("route_median_elapsed")
)

profile = silver.agg(
    F.avg("sched_deps_origin_hour").alias("avg_deps_hour"),
    F.max("sched_deps_origin_hour").alias("max_deps_hour"),
    F.avg("dep_bank_density").alias("avg_bank"),
    F.max("dep_bank_density").alias("max_bank"),
    F.avg("dep_sequence_in_day").alias("avg_seq"),
    F.max("dep_sequence_in_day").alias("max_seq"),
    F.avg("schedule_padding").alias("avg_padding"),
    F.expr("percentile_approx(schedule_padding, 0.95)").alias("p95_padding"),
).first()

print("Congestion features built from the schedule alone:")
print(f"  departures per origin-hour : mean {profile['avg_deps_hour']:.1f}, "
      f"max {profile['max_deps_hour']:,}")
print(f"  bank density (+/-60 min)   : mean {profile['avg_bank']:.1f}, "
      f"max {profile['max_bank']:,}")
print(f"  carrier departure sequence : mean {profile['avg_seq']:.1f}, "
      f"max {profile['max_seq']:,}")
print(f"  schedule padding (min)     : mean {profile['avg_padding']:+.1f}, "
      f"p95 {profile['p95_padding']:+.0f}")


Congestion features built from the schedule alone:
  departures per origin-hour : mean 3.3, max 21
  bank density (+/-60 min)   : mean 5.2, max 32
  carrier departure sequence : mean 5.9, max 100
  schedule padding (min)     : mean +0.3, p95 +11


### Do they separate? Check before spending a training run on them.

A feature family justified by a decomposition is still only a hypothesis. The cheapest
possible test is whether the delay rate moves across each feature's range — if it is flat
here it will be flat in the model, and it is better to find that out now than after a full
cross-validated search.

This is a marginal view and says nothing about what survives once the other features are
present, but a feature that cannot separate on its own is unlikely to earn its place later.


In [0]:
labeled_silver = (
    silver
    .filter(col("arrival_delay").isNotNull())
    .withColumn("is_delayed",
                (col("arrival_delay") >= config.DELAY_THRESHOLD_MINUTES).cast("double"))
)
base_rate = labeled_silver.agg(F.avg("is_delayed")).first()[0]
print(f"Base delay rate: {base_rate:.2%}\n")

CHECKS = [
    ("sched_deps_origin_hour", [0, 10, 20, 30, 40, 60, 10_000]),
    ("dep_bank_density", [0, 20, 40, 60, 80, 120, 10_000]),
    ("dep_sequence_in_day", [0, 1, 2, 4, 8, 16, 10_000]),
    ("schedule_padding", [-10_000, -15, -5, 5, 15, 30, 10_000]),
]

for feature, edges in CHECKS:
    bucketed = labeled_silver.withColumn(
        "bucket",
        F.when(col(feature) <= edges[1], F.lit(f"<= {edges[1]}"))
         .when(col(feature) <= edges[2], F.lit(f"{edges[1]}-{edges[2]}"))
         .when(col(feature) <= edges[3], F.lit(f"{edges[2]}-{edges[3]}"))
         .when(col(feature) <= edges[4], F.lit(f"{edges[3]}-{edges[4]}"))
         .when(col(feature) <= edges[5], F.lit(f"{edges[4]}-{edges[5]}"))
         .otherwise(F.lit(f"> {edges[5]}")),
    )
    stats = (
        bucketed.groupBy("bucket")
        .agg(F.avg("is_delayed").alias("delay_rate"), F.count("*").alias("n"))
        .toPandas()
    )
    order = [f"<= {edges[1]}", f"{edges[1]}-{edges[2]}", f"{edges[2]}-{edges[3]}",
             f"{edges[3]}-{edges[4]}", f"{edges[4]}-{edges[5]}", f"> {edges[5]}"]
    stats["_o"] = stats["bucket"].apply(lambda b: order.index(b) if b in order else 99)
    stats = stats.sort_values("_o")
    stats = stats[stats["n"] > 0]

    spread = stats["delay_rate"].max() - stats["delay_rate"].min()
    print(f"{feature}  (spread {spread:.1%})")
    for _, r in stats.iterrows():
        bar = "#" * int(r["delay_rate"] * 100)
        print(f"    {r['bucket']:>12}  {r['delay_rate']:6.2%}  n={int(r['n']):>9,}  {bar}")
    print()

print("Compare each spread against the ~19.6% spread hour-of-day showed in 02_eda §3.")
print("A feature that moves the rate by a point or two is the holiday flags again;")
print("one that moves it by ten is worth the training run.")


Base delay rate: 19.88%

sched_deps_origin_hour  (spread 69.0%)
           <= 10  19.86%  n=2,420,307  ###################
           10-20  20.63%  n=   43,654  ####################
           20-30  88.89%  n=       18  ########################################################################################

dep_bank_density  (spread 1.9%)
           <= 20  19.88%  n=2,456,659  ###################
           20-40  17.96%  n=    7,320  #################

dep_sequence_in_day  (spread 11.5%)
            <= 1  15.97%  n=  811,816  ###############
             1-2  18.07%  n=  367,575  ##################
             2-4  19.68%  n=  380,977  ###################
             4-8  21.60%  n=  379,644  #####################
            8-16  25.25%  n=  329,275  #########################
            > 16  27.52%  n=  194,692  ###########################

schedule_padding  (spread 8.0%)
          <= -15  25.83%  n=   38,990  #########################
          -15--5  22.19%  n=  504,152  #

## Write, constrain, compact

Three things happen here that the branch previously did none of.

**Table properties.** `optimizeWrite` and `autoCompact` make Delta produce
reasonably-sized files on write instead of leaving a pile of small ones for a later
`OPTIMIZE` to clean up.

**`CHECK` constraints.** The filters above are invariants — 2020 is excluded, dates
are non-null, quarters are 1–4. Expressing them as constraints moves enforcement from
"this notebook did it correctly" to "the table will reject a write that doesn't".
That is the difference between a script and a managed table, and it is what makes a
downstream consumer able to trust the schema. Note `arrival_delay` is deliberately
*not* constrained non-null: cancelled and diverted flights legitimately have none,
and Gold filters them.

**`OPTIMIZE ... ZORDER BY`.** Every downstream reader filters on `flight_year` —
`04_gold` labels off it, `05_train` splits the CV, threshold, and test windows on it.
Z-ordering co-locates those rows so those filters skip files instead of scanning.
Liquid clustering (`CLUSTER BY`) is the newer answer and would be preferable on a
paid workspace; ZORDER is used here because it is dependable on Free Edition.


In [0]:
(
    silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.SILVER)
)

silver_count = spark.table(config.SILVER).count()
print(f"Silver rows: {silver_count:,}")
print(f"Silver columns: {len(spark.table(config.SILVER).columns)}")


Silver rows: 2,520,650
Silver columns: 30


In [0]:
spark.sql(f"""
    ALTER TABLE {config.SILVER} SET TBLPROPERTIES (
        delta.autoOptimize.optimizeWrite = true,
        delta.autoOptimize.autoCompact  = true
    )
""")

# Idempotent: drop before add so re-running the notebook does not fail.
CONSTRAINTS = {
    "flight_date_present": "flight_date IS NOT NULL",
    "covid_year_excluded": "flight_year <> 2020",
    "quarter_in_range": "quarter BETWEEN 1 AND 4",
    "day_of_week_in_range": "day_of_week BETWEEN 1 AND 7",
    "delay_threshold_sane": "crs_elapsed_time IS NULL OR crs_elapsed_time > 0",
}
for name, expr in CONSTRAINTS.items():
    spark.sql(f"ALTER TABLE {config.SILVER} DROP CONSTRAINT IF EXISTS {name}")
    spark.sql(f"ALTER TABLE {config.SILVER} ADD CONSTRAINT {name} CHECK ({expr})")
    print(f"  constraint {name:<22} {expr}")

print("\nEvery constraint validated against the data already in the table —")
print("ALTER TABLE ADD CONSTRAINT fails if any existing row violates it, so the")
print("fact that this cell completed is itself the data-quality assertion.")


  constraint flight_date_present    flight_date IS NOT NULL
  constraint covid_year_excluded    flight_year <> 2020
  constraint quarter_in_range       quarter BETWEEN 1 AND 4
  constraint day_of_week_in_range   day_of_week BETWEEN 1 AND 7
  constraint delay_threshold_sane   crs_elapsed_time IS NULL OR crs_elapsed_time > 0

Every constraint validated against the data already in the table —
ALTER TABLE ADD CONSTRAINT fails if any existing row violates it, so the
fact that this cell completed is itself the data-quality assertion.


In [0]:
spark.sql(f"OPTIMIZE {config.SILVER} ZORDER BY (flight_year, origin_airport_code)")
display(spark.sql(f"DESCRIBE HISTORY {config.SILVER}").select(
    "version", "timestamp", "operation", "operationMetrics"
).limit(10))


version,timestamp,operation,operationMetrics
19,2026-09-10T20:53:35.000Z,ADD CONSTRAINT,Map()
18,2026-09-10T20:53:34.000Z,DROP CONSTRAINT,Map()
17,2026-09-10T20:53:33.000Z,ADD CONSTRAINT,Map()
16,2026-09-10T20:53:31.000Z,DROP CONSTRAINT,Map()
15,2026-09-10T20:53:30.000Z,ADD CONSTRAINT,Map()
14,2026-09-10T20:53:29.000Z,DROP CONSTRAINT,Map()
13,2026-09-10T20:53:28.000Z,ADD CONSTRAINT,Map()
12,2026-09-10T20:53:26.000Z,DROP CONSTRAINT,Map()
11,2026-09-10T20:53:25.000Z,ADD CONSTRAINT,Map()
10,2026-09-10T20:53:23.000Z,DROP CONSTRAINT,Map()


The history above is the audit trail: every write, constraint change, and compaction
is a numbered version. It is also what makes the pipeline debuggable after the fact —
`SELECT * FROM silver_flights VERSION AS OF n` reads the table as it stood before a
change, which is how you answer "did this number move because the model changed or
because the data did?"


In [0]:
history = spark.sql(f"DESCRIBE HISTORY {config.SILVER}")
current = history.agg(F.max("version")).first()[0]
print(f"Current Silver version: {current}")
print(f"Time travel is available: SELECT * FROM {config.SILVER} VERSION AS OF {current}")

quality = spark.table(config.SILVER).agg(
    F.count("*").alias("rows"),
    F.countDistinct("flight_date").alias("distinct_dates"),
    F.sum(F.when(col("arrival_delay").isNull(), 1).otherwise(0)).alias("null_arrival_delay"),
    F.min("flight_date").alias("first_date"),
    F.max("flight_date").alias("last_date"),
).first()

print(f"\n  rows                 {quality['rows']:,}")
print(f"  distinct dates       {quality['distinct_dates']:,}")
print(f"  null arrival_delay   {quality['null_arrival_delay']:,} "
      f"({quality['null_arrival_delay'] / quality['rows']:.2%})  <- cancelled/diverted")
print(f"  date range           {quality['first_date']} -> {quality['last_date']}")


Current Silver version: 19
Time travel is available: SELECT * FROM workspace.flights.silver_flights VERSION AS OF 19

  rows                 2,520,650
  distinct dates       1,338
  null arrival_delay   56,671 (2.25%)  <- cancelled/diverted
  date range           2019-01-01 -> 2023-08-31
